In [ ]:
import os
import shutil
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

folder_name = 'biblioteca'

# 1. Reset directory
if os.path.exists(folder_name):
    shutil.rmtree(folder_name)

os.makedirs(folder_name)

# 2. Load dataset
ut_df = pd.read_csv('clean/user_taggedartists_clean.csv')

# 3. Group by artistID and write tags to individual files
file_extension = '.txt'

for artist_id, group in ut_df.groupby('artistID'):
    filepath = os.path.join(folder_name, f"{artist_id}{file_extension}")
    # Extract canonical tags and write them (one per line)
    tags = group['canonical_tag'].dropna().tolist()
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write('\n'.join(tags) + '\n')

print(f"Successfully populated files for {ut_df['artistID'].nunique()} artists in '{folder_name}'.")


# ---------------------------------------------------------
# 4. Build TF-IDF Sparse Matrix
# ---------------------------------------------------------

# Collect file paths and retain order of artist IDs
file_paths = []
artist_ids = []

for filename in sorted(os.listdir(folder_name)):
    if filename.endswith(file_extension):
        file_paths.append(os.path.join(folder_name, filename))
        artist_ids.append(filename.replace(file_extension, ''))


def newline_tokenizer(text):
    return [line.strip() for line in text.splitlines() if line.strip()]

# Initialize TfidfVectorizer: split strictly by newline characters
vectorizer = TfidfVectorizer(
    input='filename',
    tokenizer=newline_tokenizer,
    lowercase=True,
    token_pattern=None
)

# Fit and transform to create the sparse matrix
tfidf_matrix = vectorizer.fit_transform(file_paths)

# Inspect the result
print("\n--- TF-IDF Matrix Built ---")
print(f"Matrix shape (Artists x Unique Tags): {tfidf_matrix.shape}")
print(f"Matrix type: {type(tfidf_matrix)}")  # scipy.sparse._csr.csr_matrix



# Get feature names (tags)
feature_names = vectorizer.get_feature_names_out()

print(f"Total Unique Tags: {len(feature_names)}")
print("First 10 tags:", feature_names[:10])


target_artist_id = "52"  # Change to any artistID in your dataset

if target_artist_id in artist_ids:
    # 1. Find row index in the list
    row_idx = artist_ids.index(target_artist_id)
    
    # 2. Extract sparse row using .getrow() to avoid direct bracket indexing errors
    artist_row = tfidf_matrix.getrow(row_idx)
    
    # 3. Get non-zero column indices and their corresponding TF-IDF values
    _, col_indices = artist_row.nonzero()
    scores = artist_row.data
    
    # 4. Map back to tag names
    feature_names = vectorizer.get_feature_names_out()
    
    df_artist = pd.DataFrame({
        'tag': feature_names[col_indices],
        'tfidf': scores
    }).sort_values(by='tfidf', ascending=False)

    print(f"\n--- Top Tags for Artist ID: {target_artist_id} ---")
    print(df_artist.head(10).to_string(index=False))
else:
    print(f"Artist ID {target_artist_id} not found.")


import textwrap

# Join all names into a single string separated by spaces (or commas)
full_text = " ".join(feature_names)

# Wrap to a maximum of 140 characters per line
wrapped_lines = textwrap.wrap(full_text, width=140)

for line in wrapped_lines:
    print(line)

# 147 Total Unique Tags: 9482
# 154                    9476
# 155                    9475
# 161                    9470
# 162                    9469
# 163                    9468

def cleanup_saved_files():
    """Deletes the TF-IDF matrix, vectorizer, and artist IDs files if they exist."""
    files_to_delete = [
        'tfidf_matrix.npz',
        'vectorizer.joblib',
        'artist_ids.json'
    ]
    for file_path in files_to_delete:
        if os.path.exists(file_path):
            os.remove(file_path)
            print(f"Deleted: {file_path}")
        else:
            print(f"Not found (skipped): {file_path}")

cleanup_saved_files()

import json
import joblib
from scipy.sparse import save_npz
# 1. Save SciPy sparse matrix (.npz is faster & more compact than pickle)
save_npz('tfidf_matrix.npz', tfidf_matrix)
# 2. Save the fitted TfidfVectorizer
joblib.dump(vectorizer, 'vectorizer.joblib')
# 3. Save artist_ids list to maintain exact row ordering
with open('artist_ids.json', 'w', encoding='utf-8') as f:
    json.dump(artist_ids, f)
print("Static artifacts successfully saved to disk!")

Successfully populated files for 12130 artists in 'biblioteca'.

--- TF-IDF Matrix Built ---
Matrix shape (Artists x Unique Tags): (12130, 9468)
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Total Unique Tags: 9468
First 10 tags: ['-pearl fashion music' '0 play yet' '00' '007' '00s rock' '1' '1008'
 '10101' '10s' '111']

--- Top Tags for Artist ID: 52 ---
             tag    tfidf
        trip-hop 0.747383
        chillout 0.406566
       downtempo 0.336254
      electronic 0.189225
 female vocalist 0.132785
          lounge 0.111550
female vovalists 0.103383
         the end 0.087369
         science 0.084307
      drug music 0.082665
-pearl fashion music 0 play yet 00 007 00s rock 1 1008 10101 10s 111 112 12 stones 1200 micrograms 1200 mics 1337 1337 guitar players 1488
18 hits 1900 1900s 1940s 1950s 1960s 1963 1964 1965 1966 1967 1968 1969 1970 1970s 1971 1973 1974 1975 1976 1977 1978 1979 1979 songs 1980
1980s 1981 1982 1983 1984 1985 1986 1987 1988 1989 1990 1990 in at nutsh